# REST API check (API v3, `docs/api.md`)
Run top to bottom against a device. Gated routes need a login: set `CAN_SNIFFER_TOKEN` (a session token) or `CAN_SNIFFER_PIN` (the device is claimed), or use a dev image with `dev_bypass`. Without either, gated cells print `SKIPPED (401)` instead of failing.

Not run by default: `RUN_CLEAR_DTC` (clears codes on a real car), `RUN_WIFI_SCAN` (can drop Wi-Fi clients), `RUN_REBOOT`.

In [1]:
import os
import time
import requests
from rich import print


can_sniffer_ip = os.environ.get("CAN_SNIFFER_URL", "http://can-sniffer.local")
base = f"{can_sniffer_ip}/api/v1"
token = os.environ.get("CAN_SNIFFER_TOKEN", "")
headers = {}

RUN_CLEAR_DTC = False  # sends OBD mode 0x04 to 0x7DF: clears codes on a real vehicle
RUN_WIFI_SCAN = False  # off-channel scan can disconnect clients
RUN_REBOOT = False


def call(method, path, **kw):
    """Request /api/v1<path> with the session headers; kw['headers'] extend them."""
    h = {**headers, **kw.pop("headers", {})}
    return requests.request(method, f"{base}{path}", headers=h, timeout=15, **kw)


def check(response, expected, label=""):
    """Print pass/fail for the expected status. A gated route without login is skipped, not failed."""
    codes = expected if isinstance(expected, (list, tuple)) else [expected]
    if response.status_code == 401 and 401 not in codes:
        print(f"[bold yellow]SKIPPED[/bold yellow] {label} (401: no login, set CAN_SNIFFER_TOKEN or CAN_SNIFFER_PIN)")
        return False
    ok = response.status_code in codes
    print(f"[bold {'green' if ok else 'red'}]{'Test Passed' if ok else 'Test Failed'}[/bold {'green' if ok else 'red'}] {label} -> {response.status_code} (expected {expected})")
    body = response.text[:600]
    if body:
        print("Response:", body)
    return ok

In [2]:
# GET /

try:
    response = requests.get(can_sniffer_ip, timeout=10)
    if response.status_code == 200:
        print("CAN Sniffer is reachable!")
    else:
        print("CAN Sniffer is not reachable.")
except requests.RequestException as e:
    print("Error connecting to CAN Sniffer:", e)

CAN Sniffer is reachable!

In [3]:
# GET /api/v1/auth  (open). dev_bypass: every gated route passes without a token.

response = call("GET", "/auth")
check(response, 200, "GET /auth")
auth = response.json()
print("state:", auth.get("state"), "dev_bypass:", auth.get("dev_bypass", False), "authenticated:", auth.get("authenticated"))

if not token and os.environ.get("CAN_SNIFFER_PIN") and auth.get("state") == "claimed":
    # POST /api/v1/auth/login
    login = call("POST", "/auth/login", json={"pin": os.environ["CAN_SNIFFER_PIN"]})
    if check(login, 200, "POST /auth/login"):
        token = login.json()["token"]
if token:
    headers["Authorization"] = f"Bearer {token}"
logged_in = bool(token) or bool(auth.get("dev_bypass"))
print("gated routes usable:", logged_in)

Test Passed GET /auth -> 200 (expected 200)

Response: {"state":"claimed","authenticated":false,"retry_after_s":0,"claim_expires_in_s":0,"ap_open":false}

state: claimed dev_bypass: False authenticated: False

gated routes usable: False

In [4]:
# Guards: Host, Origin, Content-Type (before auth and rate limit)

check(call("GET", "/system", headers={"Host": "evil.example"}), 403, "bad Host -> BAD_HOST")
check(call("GET", "/system", headers={"Origin": "http://evil.example"}), 403, "bad Origin -> BAD_ORIGIN")
check(call("PATCH", "/obd2", data='{"auto_poll":false}', headers={"Content-Type": "text/plain"}), 415, "text/plain body -> BAD_CONTENT_TYPE")
check(call("GET", "/dtcs"), 404, "unknown path")
check(call("POST", "/obd2", json={"pid_def_path": "/sdcard/config/pid_def.json"}), 405, "POST /obd2 is not a route")

Test Passed bad Host -> BAD_HOST -> 403 (expected 403)

Response: {"status":"error","reason":"Host not allowed","code":"BAD_HOST"}

Test Passed bad Origin -> BAD_ORIGIN -> 403 (expected 403)

Response: {"status":"error","reason":"Origin not allowed","code":"BAD_ORIGIN"}

Test Passed text/plain body -> BAD_CONTENT_TYPE -> 415 (expected 415)

...

Test Passed unknown path -> 404 (expected 404)

Response: Nothing matches the given URI

Test Passed POST /obd2 is not a route -> 405 (expected 405)

Response: Specified method is invalid for this resource

True

In [5]:
# GET /api/v1/system  (api_version 3)

response = call("GET", "/system")
ok = check(response, 200, "GET /system")
print("api_version == 3:", response.json().get("api_version") == 3, "| sd_card_detected:", response.json().get("sd_card_detected"))
sd_ok = response.json().get("sd_card_detected", False)

Test Passed GET /system -> 200 (expected 200)

Response: {"api_version":3,"app_version":"1.2.4","uptime_s":198,"restart_reason":"USB 
reset","mac":"D0:CF:13:1C:18:38","state":3,"battery_voltage":0.899999976158142,"sd_card_detected":true,"component_s
tatus":[{"name":"SETTINGS","status":"ESP_OK"},{"name":"POWER_MANAGEMENT","status":"ESP_OK"},{"name":"FLASH_FILESYST
EM","status":"ESP_OK"},{"name":"SD_CARD","status":"ESP_OK"},{"name":"BATTERY","status":"ESP_OK"},{"name":"CAN","sta
tus":"ESP_OK"},{"name":"OBD","status":"ESP_OK"},{"name":"WIFI","status":"ESP_OK"},{"name":"WEBSERVER","status":"ESP
_OK"}],"heap_free":8423963,"heap_min_free":8388512,"tasks":[{"na

api_version == 3: True | sd_card_detected: True

In [6]:
# GET /api/v1/can_bus

check(call("GET", "/can_bus"), 200, "GET /can_bus")

Test Passed GET /can_bus -> 200 (expected 200)

Response: 
{"state":"connected","bitrate":500000,"rs_mode":"high_speed","debug_mode":true,"initialized":true,"bus_connected":t
rue,"node_status":{"twai_error_state":"error_active","tx_error_count":0,"rx_error_count":0}}

True

In [7]:
# GET /api/v1/obd2

response = call("GET", "/obd2")
check(response, 200, "GET /obd2")
orig_auto_poll = response.json().get("auto_poll", False)

Test Passed GET /obd2 -> 200 (expected 200)

Response: 
{"auto_poll":false,"polling":false,"pid_initialized":true,"pid_def_count":10,"pid_data_count":10,"poll_task_utiliza
tion":0.000712268694769591,"pid_def_path":"/sdcard/config/pid_def.json","dtc_desc_path":"/sdcard/config/dtcs.bin","
supported_pids":{"count":224,"groups":[4294967295,4294967295,4294967295,4294967295,4294967295,4294967295,4294967295
]}}

In [8]:
# PATCH /api/v1/obd2 {"auto_poll": bool}  (replaces POST /req/auto_poll) -> 200 {auto_poll, polling}
# Sets the value to what it already is, then restores it at the end.

response = call("PATCH", "/obd2", json={"auto_poll": orig_auto_poll})
if check(response, 200, "PATCH /obd2") and response.status_code == 200:
    print("auto_poll echoed:", response.json().get("auto_poll") == orig_auto_poll)
    check(call("GET", "/obd2"), 200, "GET /obd2 after PATCH")

# bad bodies -> 400
check(call("PATCH", "/obd2", json={}), 400, "PATCH /obd2 empty")
check(call("PATCH", "/obd2", json={"auto_poll": 1}), 400, "PATCH /obd2 non-boolean")
check(call("PATCH", "/obd2", json={"auto_poll": True, "x": 1}), 400, "PATCH /obd2 extra key")

Test Passed PATCH /obd2 -> 200 (expected 200)

Response: {"auto_poll":false,"polling":false}

auto_poll echoed: True

Test Passed GET /obd2 after PATCH -> 200 (expected 200)

Response: 
{"auto_poll":false,"polling":false,"pid_initialized":true,"pid_def_count":10,"pid_data_count":10,"poll_task_utiliza
tion":0.0006054284167476,"pid_def_path":"/sdcard/config/pid_def.json","dtc_desc_path":"/sdcard/config/dtcs.bin","su
pported_pids":{"count":224,"groups":[4294967295,4294967295,4294967295,4294967295,4294967295,4294967295,4294967295]}
}

...

Test Passed PATCH /obd2 non-boolean -> 400 (expected 400)

Response: {"status":"error","reason":"Body must be {\"auto_poll\": true|false}","code":"BAD_REQUEST"}

Test Passed PATCH /obd2 extra key -> 400 (expected 400)

Response: {"status":"error","reason":"Body must be {\"auto_poll\": true|false}","code":"BAD_REQUEST"}

True

In [9]:
# GET /api/v1/settings  (gated) -> {wifi, can, system}; passwords are never returned

response = call("GET", "/settings")
if check(response, 200, "GET /settings") and response.status_code == 200:
    settings = response.json()
    print("sections:", sorted(settings), "| no password fields:", "password" not in settings.get("wifi", {}))

SKIPPED GET /settings (401: no login, set CAN_SNIFFER_TOKEN or CAN_SNIFFER_PIN)

In [10]:
# PATCH /api/v1/settings  (gated). Partial: only the given fields change; saved to NVS, applied after reboot.
# Writes the current value back, so nothing changes. Do not touch wifi fields here.

if logged_in and response.status_code == 200:
    current = settings["can"]["debug"]
    patched = call("PATCH", "/settings", json={"can": {"debug": current}})
    check(patched, 200, "PATCH /settings can.debug (same value)")
    if patched.status_code == 200:
        print("restart_required:", patched.json().get("restart_required"))
    check(call("PATCH", "/settings", json={"nosuch": {"a": 1}}), 422, "PATCH /settings unknown section")
    check(call("PATCH", "/settings", json={"can": {"bitrate": 12345}}), 400, "PATCH /settings invalid bitrate")
else:
    print("[bold yellow]SKIPPED[/bold yellow] PATCH /settings (needs login)")

SKIPPED PATCH /settings (needs login)

In [11]:
# GET /api/v1/sd_card/info

response = call("GET", "/sd_card/info")
check(response, 200, "GET /sd_card/info")
sd_mounted = response.status_code == 200 and response.json().get("is_mounted", False)

Test Passed GET /sd_card/info -> 200 (expected 200)

Response: 
{"name":"ISTAD","mount_path":"/sdcard","capacity":29823,"used_space_mb":0,"max_freq_mhz":20,"is_sdio":false,"is_mmc
":false,"is_mounted":true,"is_present":true}

In [12]:
# GET /api/v1/sd_card/tree  (gated)

check(call("GET", "/sd_card/tree", params={"depth": 1}), 200, "GET /sd_card/tree")

SKIPPED GET /sd_card/tree (401: no login, set CAN_SNIFFER_TOKEN or CAN_SNIFFER_PIN)

False

In [13]:
# PUT /api/v1/sd_card/file/<path>  (gated): 201 created, 200 overwritten. Throwaway file, deleted below.

tmp_file = "config/api_check_tmp.txt"
tmp_copy = "config/api_check_tmp_copy.txt"
tmp_dir = "api_check_dir/"
tmp_body = b"rest_api_check\n"
octet = {"Content-Type": "application/octet-stream"}

check(call("PUT", f"/sd_card/file/{tmp_file}", data=tmp_body, headers=octet), 201, "PUT file (create)")
check(call("PUT", f"/sd_card/file/{tmp_file}", data=tmp_body, headers=octet), 200, "PUT file (overwrite)")
check(call("PUT", f"/sd_card/file/{tmp_dir}", data=b"", headers=octet), 201, "PUT dir (create)")
check(call("PUT", f"/sd_card/file/{tmp_dir}", data=b"", headers=octet), 200, "PUT dir (exists)")
check(call("PUT", "/sd_card/file/a/../b.txt", data=tmp_body, headers=octet), 400, "PUT bad path")

SKIPPED PUT file (create) (401: no login, set CAN_SNIFFER_TOKEN or CAN_SNIFFER_PIN)

SKIPPED PUT file (overwrite) (401: no login, set CAN_SNIFFER_TOKEN or CAN_SNIFFER_PIN)

SKIPPED PUT dir (create) (401: no login, set CAN_SNIFFER_TOKEN or CAN_SNIFFER_PIN)

SKIPPED PUT dir (exists) (401: no login, set CAN_SNIFFER_TOKEN or CAN_SNIFFER_PIN)

SKIPPED PUT bad path (401: no login, set CAN_SNIFFER_TOKEN or CAN_SNIFFER_PIN)

False

In [14]:
# GET /api/v1/sd_card/file/<path>  (gated)

response = call("GET", f"/sd_card/file/{tmp_file}")
if check(response, 200, "GET file") and response.status_code == 200:
    print("content matches:", response.content == tmp_body)
check(call("GET", "/sd_card/file/config/api_check_missing.txt"), 404, "GET missing file")

SKIPPED GET file (401: no login, set CAN_SNIFFER_TOKEN or CAN_SNIFFER_PIN)

SKIPPED GET missing file (401: no login, set CAN_SNIFFER_TOKEN or CAN_SNIFFER_PIN)

False

In [15]:
# POST /api/v1/sd_card/copy  (gated): relative paths, overwrites destination

body = {"source": tmp_file, "destination": tmp_copy}
check(call("POST", "/sd_card/copy", json=body), 200, "POST /sd_card/copy")
response = call("GET", f"/sd_card/file/{tmp_copy}")
if response.status_code == 200:
    print("copy matches:", response.content == tmp_body)
check(call("POST", "/sd_card/copy", json={"source": tmp_file, "destination": tmp_file}), 400, "copy onto itself")
check(call("POST", "/sd_card/copy", json={"source": "config/api_check_missing.txt", "destination": tmp_copy}), 404, "copy missing source")

SKIPPED POST /sd_card/copy (401: no login, set CAN_SNIFFER_TOKEN or CAN_SNIFFER_PIN)

SKIPPED copy onto itself (401: no login, set CAN_SNIFFER_TOKEN or CAN_SNIFFER_PIN)

SKIPPED copy missing source (401: no login, set CAN_SNIFFER_TOKEN or CAN_SNIFFER_PIN)

False

In [16]:
# DELETE /api/v1/sd_card/file/<path>  (gated) -> 204, no body. Removes the throwaway files.

check(call("DELETE", f"/sd_card/file/{tmp_file}"), 204, "DELETE file")
check(call("DELETE", f"/sd_card/file/{tmp_copy}"), 204, "DELETE copy")
check(call("DELETE", f"/sd_card/file/{tmp_dir}"), 204, "DELETE dir")
check(call("DELETE", f"/sd_card/file/{tmp_file}"), 404, "DELETE already gone")

SKIPPED DELETE file (401: no login, set CAN_SNIFFER_TOKEN or CAN_SNIFFER_PIN)

SKIPPED DELETE copy (401: no login, set CAN_SNIFFER_TOKEN or CAN_SNIFFER_PIN)

SKIPPED DELETE dir (401: no login, set CAN_SNIFFER_TOKEN or CAN_SNIFFER_PIN)

SKIPPED DELETE already gone (401: no login, set CAN_SNIFFER_TOKEN or CAN_SNIFFER_PIN)

False

In [17]:
# GET /api/v1/pid_def

response = call("GET", "/pid_def")
check(response, 200, "GET /pid_def")
pid_defs = response.json().get("data", []) if response.status_code == 200 else []

Test Passed GET /pid_def -> 200 (expected 200)

Response: {"data":[{"pid":4,"mode":1,"id":2015,"length":2,"name":"Engine Load","unit":"%","description":"Calculated
engine 
load","minValue":0,"maxValue":100,"priority":2,"update_interval_ms":256,"color":16096779,"icon":"gauge","formula":"
A * 100 / 255"},{"pid":5,"mode":1,"id":2015,"length":2,"name":"Coolant Temp","unit":"°C","description":"Engine 
coolant 
temperature","minValue":-40,"maxValue":215,"priority":3,"update_interval_ms":4096,"color":15680580,"icon":"thermome
ter","formula":"A - 40"},{"pid":12,"mode":1,"id":2015,"length":2,"name":"Engine 
RPM","unit":"rpm","description":"Engine speed","minValue":

In [18]:
# GET /api/v1/pid_def/<pid>  -> one bare PIDDef object

pid = pid_defs[0]["pid"] if pid_defs else 4
response = call("GET", f"/pid_def/{pid}")
if check(response, 200, f"GET /pid_def/{pid}") and response.status_code == 200:
    print("bare object, pid matches:", response.json().get("pid") == pid and "data" not in response.json())
check(call("GET", "/pid_def/foo"), 400, "GET /pid_def/foo")
check(call("GET", "/pid_def/65535"), 404, "GET /pid_def/65535 (undefined)")

Test Passed GET /pid_def/4 -> 200 (expected 200)

Response: {"pid":4,"mode":1,"id":2015,"length":2,"name":"Engine Load","unit":"%","description":"Calculated engine 
load","minValue":0,"maxValue":100,"priority":2,"update_interval_ms":256,"color":16096779,"icon":"gauge","formula":"
A * 100 / 255"}

bare object, pid matches: True

Test Passed GET /pid_def/foo -> 400 (expected 400)

Response: {"status":"error","reason":"Invalid PID requested","code":"BAD_REQUEST"}

Test Passed GET /pid_def/65535 (undefined) -> 404 (expected 404)

Response: {"status":"error","reason":"Not found","code":"NOT_FOUND"}

True

In [19]:
# POST /api/v1/pid_def  (gated): add a throwaway PID (60000), then DELETE it -> 204
# Required: id, mode, pid, name, formula, interval. len is an alias of length.

test_pid = 60000
payload = [
  {
    "id": 2015,
    "mode": 1,
    "pid": test_pid,
    "len": 1,
    "name": "API check",
    "unit": "%",
    "desc": "Throwaway PID from rest_api_check",
    "formula": "A * 100 / 255",
    "minV": 0.0,
    "maxV": 100.0,
    "priority": 2,
    "interval": 1000,
    "color": 16096779,
    "icon": "gauge"
  }
]

response = call("POST", "/pid_def", json=payload)
if check(response, 200, "POST /pid_def") and response.status_code == 200:
    print("added:", response.json().get("added"))
    check(call("GET", f"/pid_def/{test_pid}"), 200, "GET new PID")

SKIPPED POST /pid_def (401: no login, set CAN_SNIFFER_TOKEN or CAN_SNIFFER_PIN)

In [20]:
# POST /api/v1/pid_def/save and /load  (gated), throwaway file; the default pid_def.json is not touched.
# load replaces the whole set with the file, so the throwaway PID stays until it is deleted below.

tmp_json = "/sdcard/config/api_check_tmp.json"
check(call("POST", "/pid_def/save", json={"pid_def_path": tmp_json}), 200, "POST /pid_def/save")
check(call("POST", "/pid_def/load", json={"pid_def_path": tmp_json}), 200, "POST /pid_def/load")
check(call("POST", "/pid_def/save", json={"pid_def_path": "/etc/x.json"}), 400, "save outside /sdcard")
check(call("POST", "/pid_def/load", json={"pid_def_path": "/sdcard/config/api_check_missing.json"}), 404, "load missing file")
check(call("DELETE", "/sd_card/file/config/api_check_tmp.json"), 204, "DELETE throwaway pid json")

SKIPPED POST /pid_def/save (401: no login, set CAN_SNIFFER_TOKEN or CAN_SNIFFER_PIN)

SKIPPED POST /pid_def/load (401: no login, set CAN_SNIFFER_TOKEN or CAN_SNIFFER_PIN)

SKIPPED save outside /sdcard (401: no login, set CAN_SNIFFER_TOKEN or CAN_SNIFFER_PIN)

SKIPPED load missing file (401: no login, set CAN_SNIFFER_TOKEN or CAN_SNIFFER_PIN)

SKIPPED DELETE throwaway pid json (401: no login, set CAN_SNIFFER_TOKEN or CAN_SNIFFER_PIN)

False

In [21]:
# DELETE /api/v1/pid_def/<pid>  (gated) -> 204

check(call("DELETE", f"/pid_def/{test_pid}"), 204, "DELETE /pid_def")
check(call("DELETE", f"/pid_def/{test_pid}"), 404, "DELETE /pid_def again")

SKIPPED DELETE /pid_def (401: no login, set CAN_SNIFFER_TOKEN or CAN_SNIFFER_PIN)

SKIPPED DELETE /pid_def again (401: no login, set CAN_SNIFFER_TOKEN or CAN_SNIFFER_PIN)

False

In [22]:
# GET /api/v1/pid_data and /pid_data/<pid> (bare object)

response = call("GET", "/pid_data")
check(response, 200, "GET /pid_data")
data = response.json().get("data", []) if response.status_code == 200 else []

pid = data[0]["pid"] if data else 4
response = call("GET", f"/pid_data/{pid}")
if check(response, 200, f"GET /pid_data/{pid}") and response.status_code == 200:
    print("bare object:", "data" not in response.json())
check(call("GET", "/pid_data/65535"), 404, "GET /pid_data/65535 (undefined)")
check(call("GET", "/pid_data/abc"), 400, "GET /pid_data/abc")

Test Passed GET /pid_data -> 200 (expected 200)

Response: 
{"data":[{"id":2015,"pid":4,"value":0,"lastUpdated":0,"isSupported":true,"isValid":false,"update_interval_ms":256},
{"id":2015,"pid":5,"value":0,"lastUpdated":0,"isSupported":true,"isValid":false,"update_interval_ms":4096},{"id":20
15,"pid":12,"value":0,"lastUpdated":0,"isSupported":true,"isValid":false,"update_interval_ms":256},{"id":1812,"pid"
:8707,"value":0,"lastUpdated":0,"isSupported":true,"isValid":false,"update_interval_ms":4096},{"id":1812,"pid":8710
,"value":0,"lastUpdated":0,"isSupported":true,"isValid":false,"update_interval_ms":4096},{"id":1862,"pid":9747,"val
ue":0,"lastUpdated":0,"is

Test Passed GET /pid_data/4 -> 200 (expected 200)

Response: {"id":2015,"pid":4,"value":0,"lastUpdated":0,"isSupported":true,"isValid":false,"update_interval_ms":256}

bare object: True

Test Passed GET /pid_data/65535 (undefined) -> 404 (expected 404)

Response: {"status":"error","reason":"Not found","code":"NOT_FOUND"}

Test Passed GET /pid_data/abc -> 400 (expected 400)

Response: {"status":"error","reason":"Invalid PID requested","code":"BAD_REQUEST"}

True

In [23]:
# POST /api/v1/req/vin, GET /api/v1/vin

check(call("POST", "/req/vin"), [200, 502, 503, 504], "POST /req/vin")
check(call("GET", "/vin"), 200, "GET /vin")

Test Passed POST /req/vin -> 200 (expected [200, 502, 503, 504])

Response: {"status":"success"}

Test Passed GET /vin -> 200 (expected 200)

Response: {"vin":"1HGBH41JXMN109186"}

True

In [24]:
# POST /api/v1/req/dtc, GET /api/v1/dtc, GET /api/v1/dtc?mode=3

check(call("POST", "/req/dtc"), [200, 503, 504], "POST /req/dtc")
check(call("GET", "/dtc"), 200, "GET /dtc")
check(call("GET", "/dtc", params={"mode": 3}), 200, "GET /dtc?mode=3")
check(call("GET", "/dtc", params={"mode": 5}), 422, "GET /dtc?mode=5 (unsupported)")
check(call("GET", "/dtc", params={"mode": -1}), 400, "GET /dtc?mode=-1")

Test Passed POST /req/dtc -> 200 (expected [200, 503, 504])

Response: {"successful_modes":[3,7,10],"failed_modes":[],"status":"success"}

Test Passed GET /dtc -> 200 (expected 200)

Response: 
{"dtcs":[{"mode":3,"type":"confirmed_dtcs","dtc":["P0143","B0260","C0234","P0300","P0408","P0506","P0101","P0113","
P1A10","P2345"],"dtc_count":10},{"mode":7,"type":"pending_dtcs","dtc":["P0143","B0260","C0234","P0300","P0408","P05
06","P1A10","P2345"],"dtc_count":8},{"mode":10,"type":"permanent_dtcs","dtc":["P0143","P0300","P0408","P1A10","P234
5"],"dtc_count":5}],"count":3,"status":"success"}

Test Passed GET /dtc?mode=3 -> 200 (expected 200)

...

Test Passed GET /dtc?mode=5 (unsupported) -> 422 (expected 422)

Response: {"dtcs":[],"count":0,"status":"error","reason":"Unsupported DTC mode","code":"UNPROCESSABLE"}

Test Passed GET /dtc?mode=-1 -> 400 (expected 400)

Response: {"status":"error","reason":"Invalid DTC mode","code":"BAD_REQUEST"}

True

In [25]:
# GET /api/v1/dtc/descriptions?codes=...  (was GET /dtc?codes=). Upper-case codes, 1-30.

response = call("GET", "/dtc/descriptions", params={"codes": "P1124,P0138,C0234"})
if check(response, [200, 404], "GET /dtc/descriptions") and response.status_code == 200:
    print("dtc_count:", response.json().get("dtc_count"))
check(call("GET", "/dtc/descriptions"), 400, "codes missing")
check(call("GET", "/dtc/descriptions", params={"codes": "p0301"}), 400, "lower-case code")
check(call("GET", "/dtc/descriptions", params={"codes": "P0301", "x": 1}), 400, "extra query key")

Test Passed GET /dtc/descriptions -> 200 (expected [200, 404])

Response: {"status":"success","dtc_count":3,"dtcs":[{"dtc":"P1124","description":"Manifold Absolute Temperature 
Circuit High Input"},{"dtc":"P0138","description":"O2 Sensor Circuit Low Voltage (Bank 1 Sensor 
2)"},{"dtc":"C0234","description":"Description not found"}]}

dtc_count: 3

Test Passed codes missing -> 400 (expected 400)

Response: {"status":"error","reason":"Invalid DTC query","code":"BAD_REQUEST"}

Test Passed lower-case code -> 400 (expected 400)

Response: {"status":"error","reason":"Invalid DTC code","code":"BAD_REQUEST"}

Test Passed extra query key -> 400 (expected 400)

Response: {"status":"error","reason":"Invalid DTC query","code":"BAD_REQUEST"}

True

In [26]:
# POST /api/v1/req/clear_dtc  (gated)
# SKIPPED by default: clears codes on a real vehicle. Needs explicit approval each time.

if RUN_CLEAR_DTC:
    check(call("POST", "/req/clear_dtc"), [200, 503, 504], "POST /req/clear_dtc")
else:
    print("[bold yellow]SKIPPED[/bold yellow] POST /req/clear_dtc (RUN_CLEAR_DTC is False)")

SKIPPED POST /req/clear_dtc (RUN_CLEAR_DTC is False)

In [27]:
# POST then GET /api/v1/wifi/scan  (gated). Off by default: the scan can disconnect Wi-Fi clients.

if RUN_WIFI_SCAN:
    check(call("POST", "/wifi/scan"), 202, "POST /wifi/scan")
    time.sleep(4)
    check(call("GET", "/wifi/scan"), 200, "GET /wifi/scan")
else:
    print("[bold yellow]SKIPPED[/bold yellow] /wifi/scan (RUN_WIFI_SCAN is False)")

SKIPPED /wifi/scan (RUN_WIFI_SCAN is False)

In [28]:
# Restore auto_poll to what it was at the start

check(call("PATCH", "/obd2", json={"auto_poll": orig_auto_poll}), 200, "PATCH /obd2 restore")

Test Passed PATCH /obd2 restore -> 200 (expected 200)

Response: {"auto_poll":false,"polling":false}

True

In [29]:
# POST /api/v1/system/reboot  (gated) -> 202 {"status":"success"}; the device restarts after 500 ms

if RUN_REBOOT:
    if check(call("POST", "/system/reboot"), 202, "POST /system/reboot"):
        time.sleep(3)
        for _ in range(30):
            try:
                if requests.get(f"{base}/system", timeout=2).status_code == 200:
                    print("device is back")
                    break
            except requests.RequestException:
                time.sleep(1)
else:
    print("[bold yellow]SKIPPED[/bold yellow] POST /system/reboot (RUN_REBOOT is False)")

SKIPPED POST /system/reboot (RUN_REBOOT is False)